<a href="https://colab.research.google.com/github/Gen54cw/Analitica-de-Datos-Tintaya-Daniel/blob/main/laboratorios/laboratorio2_Tintaya_Daniel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 2: Recolección de Datos y Aterrizaje en Zona Bronce

**Prof. Juan Gamarra Moreno**

**Alumno:** Tintaya Avila Daniel Eduardo

**Fecha:** 01/09/2026


In [ ]:
!pip install -q polars duckdb pyarrow requests pandas

In [ ]:
#librerias necesarias y verificación
import polars as pl
import duckdb
import pyarrow
import requests
import pandas as pd

print('polars :', pl.__version__)
print('duckdb :', duckdb.__version__)
print('pyarrow :', pyarrow.__version__)

polars : 1.35.2
duckdb : 1.3.2
pyarrow : 18.1.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Paso 1 — Crear la estructura de zonas


In [ ]:
import os, json, time, hashlib
from datetime import datetime, date, timedelta
from pathlib import Path
import requests
import polars as pl
import duckdb

# Rutas del laboratorio
DATOS = Path('/content/drive/MyDrive/Analitica/LAB02/datos')  # ajuste esta ruta si usa Google Drive
BASE = Path('lakehouse')
BRONCE = BASE / 'bronce'          # dato crudo, inmutable
PLATA = BASE / 'plata'            # dato validado y normalizado
CUARENTENA = BASE / 'cuarentena'  # registros que no pasan el contrato

for zona in (BRONCE, PLATA, CUARENTENA):
    zona.mkdir(parents=True, exist_ok=True)

print('Zonas creadas:')
for zona in (BRONCE, PLATA, CUARENTENA):
    print(' ', zona)

print('\nArchivos fuente encontrados:')
for archivo in sorted(DATOS.glob('*')):
    print(f'  {archivo.name:26s} {archivo.stat().st_size/1e6:7.2f} MB')

Zonas creadas:
  lakehouse/bronce
  lakehouse/plata
  lakehouse/cuarentena

Archivos fuente encontrados:
  bcrp_tipo_cambio.csv          0.00 MB
  clima_diario.jsonl            0.47 MB
  locales.csv                   0.00 MB
  maestro_productos.csv         0.03 MB
  ventas_pos.csv               10.54 MB


## Paso 2 — Reconocer la fuente antes de ingerirla


In [ ]:
# a) Vistazo rápido: solo las primeras 1000 filas
vistazo = pl.read_csv(DATOS / 'ventas_pos.csv', n_rows=1000)
print('Esquema inferido:')
print(vistazo.schema)

# b) Carga completa, pidiendo a Polars que interprete las fechas
ventas_raw = pl.read_csv(DATOS / 'ventas_pos.csv', try_parse_dates=True)
print(f'\nFilas: {ventas_raw.height:,} | Columnas: {ventas_raw.width}')
ventas_raw.head(5)

Esquema inferido:
Schema({'id_venta': Int64, 'fecha': String, 'id_local': Int64, 'id_producto': Int64, 'cantidad': Int64, 'importe': Float64, 'canal': String, 'medio_pago': String})

Filas: 200,600 | Columnas: 8


id_venta,fecha,id_local,id_producto,cantidad,importe,canal,medio_pago
i64,date,i64,i64,i64,f64,str,str
77288,2025-12-10,77,1203,2,30.06,"""Presencial""","""Billetera"""
129421,2025-09-09,84,1296,6,106.15,"""Delivery""","""Efectivo"""
196899,2025-10-26,74,1220,2,47.45,"""Presencial""","""Billetera"""
16250,2026-03-12,61,1092,3,229.1,"""Presencial""","""Tarjeta"""
23062,2026-02-01,73,1399,3,23.73,"""Presencial""","""Efectivo"""


In [ ]:
perfil = pl.DataFrame({
    'columna': ventas_raw.columns,
    'tipo': [str(t) for t in ventas_raw.dtypes],
    'nulos': [ventas_raw[c].null_count() for c in ventas_raw.columns],
    'distintos': [ventas_raw[c].n_unique() for c in ventas_raw.columns],
})
perfil

columna,tipo,nulos,distintos
str,str,i64,i64
"""id_venta""","""Int64""",0,200000
"""fecha""","""Date""",0,366
"""id_local""","""Int64""",0,120
"""id_producto""","""Int64""",0,898
"""cantidad""","""Int64""",0,26
"""importe""","""Float64""",4012,39656
"""canal""","""String""",0,12
"""medio_pago""","""String""",0,3


### PREGUNTAS


- El archivo tiene 200 600 filas pero `id_venta` solo tiene 200 000 valores distintos. ¿Qué explica la diferencia?

  _Respuesta: Registros duplciados en id_venta lo que indica un error en el registro
- La columna `canal` tiene 12 valores distintos, pero solo existen tres canales de venta reales. ¿Qué está ocurriendo?

  _Respuesta: Es un problema de inconsistencia, debido a errores textuales. Por ejemplo, mayusculas, espacios, o variaciones

- La columna `importe` tiene 4 010 nulos. ¿Debe eliminar esas filas, imputarlas o aislarlas? Justifique.

  _Respuesta: Deben aislarse ya que eliminarlas pierde trazabilidad, e imputar un valor monetario inventado distorsionaría los totales de venta.

- `id_producto` tiene 898 valores distintos, pero el maestro de productos solo tiene 800 registros. ¿Qué implica esto?

  _Respuesta: Implica que hay ventas con códigos de producto que no existen en el maestro, posiblemente por productos descontinuados, no registrados aún, o errores de digitación

## Paso 3 — Aterrizar en la zona bronce con procedencia

In [ ]:
def huella(ruta, bloque=1 << 20):
    """Devuelve los primeros 16 caracteres del SHA-256 del archivo.
    Permite detectar si dos ejecuciones leyeron exactamente el mismo
    contenido de origen, lo que es la base para verificar idempotencia.
    """
    h = hashlib.sha256()
    with open(ruta, 'rb') as fh:
        for trozo in iter(lambda: fh.read(bloque), b''):
            h.update(trozo)
    return h.hexdigest()[:16]

In [ ]:
def aterrizar_bronce(df, fuente, origen, particion=None):
    """Escribe un DataFrame en la zona bronce y registra la carga."""
    particion = particion or date.today().isoformat()
    destino = BRONCE / fuente / f'fecha_carga={particion}'
    destino.mkdir(parents=True, exist_ok=True)
    obtenido = datetime.now().isoformat(timespec='seconds')

    # Columnas de procedencia: acompañan a cada fila por todo el pipeline
    df = df.with_columns([
        pl.lit(fuente).alias('_fuente'),
        pl.lit(str(origen)).alias('_origen'),
        pl.lit(obtenido).alias('_obtenido_en'),
    ])

    archivo = destino / f'{fuente}.parquet'
    df.write_parquet(archivo, compression='zstd')

    # Manifiesto: una línea JSON por cada carga realizada
    registro = {
        'fuente': fuente,
        'origen': str(origen),
        'particion': particion,
        'obtenido_en': obtenido,
        'filas': df.height,
        'columnas': df.width,
        'archivo': str(archivo),
        'huella_origen': huella(origen) if Path(origen).is_file() else None,
    }
    with open(BRONCE / '_manifiesto.jsonl', 'a', encoding='utf-8') as fh:
        fh.write(json.dumps(registro, ensure_ascii=False) + '\n')

    print(f'  bronce <- {fuente}: {df.height:,} filas en {archivo}')
    return archivo

In [ ]:
aterrizar_bronce(ventas_raw, 'ventas_pos', DATOS / 'ventas_pos.csv')
aterrizar_bronce(pl.read_csv(DATOS / 'maestro_productos.csv'),
                  'maestro_productos', DATOS / 'maestro_productos.csv')
aterrizar_bronce(pl.read_csv(DATOS / 'locales.csv'),
                  'locales', DATOS / 'locales.csv')

# Revisar el manifiesto
print('\n--- Manifiesto de cargas ---')
for linea in open(BRONCE / '_manifiesto.jsonl', encoding='utf-8'):
    reg = json.loads(linea)
    print(f"{reg['fuente']:20s} {reg['filas']:>8,} filas "
          f"huella={reg['huella_origen']}")

  bronce <- ventas_pos: 200,600 filas en lakehouse/bronce/ventas_pos/fecha_carga=2026-09-02/ventas_pos.parquet
  bronce <- maestro_productos: 800 filas en lakehouse/bronce/maestro_productos/fecha_carga=2026-09-02/maestro_productos.parquet
  bronce <- locales: 120 filas en lakehouse/bronce/locales/fecha_carga=2026-09-02/locales.parquet

--- Manifiesto de cargas ---
ventas_pos            200,600 filas huella=9fd542c66e519295
maestro_productos         800 filas huella=9801f47eafce8b3e
locales                   120 filas huella=88087d8703f28356
clima_diario            2,920 filas huella=8c148f1131bf362d
bcrp_tipo_cambio          261 filas huella=7ae933ababc902d8
ventas_pos            200,600 filas huella=9fd542c66e519295
maestro_productos         800 filas huella=9801f47eafce8b3e
locales                   120 filas huella=88087d8703f28356
clima_diario            2,920 filas huella=8c148f1131bf362d
bcrp_tipo_cambio          261 filas huella=7ae933ababc902d8
ventas_pos            200,600 fil

### PREGUNTA
Explique en una celda de texto qué le dice esa coincidencia de huella sobre la idempotencia de su proceso.

Que la huella sea idéntica confirma que el archivo de origen no cambió entre una carga y otra, es decir que el contenido leído es exactamente el mismo. Sin embargo, el proceso de aterrizaje no es idempotente, porque cada ejecución agrega una nueva línea  y sobrescribe el archivo Parquet con una marca de tiempo distinta

## Paso 4 — Ingerir una fuente semiestructurada y detectar deriva de esquema


In [ ]:
brutos = [json.loads(linea)
          for linea in open(DATOS / 'clima_diario.jsonl', encoding='utf-8')]

print('Registros leídos:', len(brutos))
print('\nPRIMER registro:')
print(json.dumps(brutos[0], indent=2, ensure_ascii=False))
print('\nÚLTIMO registro:')
print(json.dumps(brutos[-1], indent=2, ensure_ascii=False))

Registros leídos: 2920

PRIMER registro:
{
  "estacion": {
    "region": "Lima",
    "codigo": "EST-01"
  },
  "fecha": "2025-09-01",
  "medicion": {
    "humedad_pct": 59,
    "precipitacion_mm": 0.0,
    "temperatura_c": 15.0
  }
}

ÚLTIMO registro:
{
  "estacion": {
    "region": "Áncash",
    "codigo": "EST-08"
  },
  "fecha": "2026-08-31",
  "medicion": {
    "humedad_pct": 71,
    "precipitacion_mm": 0.4,
    "temp_c": 14.4,
    "indice_uv": 9
  }
}


In [ ]:
conteo = {}
for reg in brutos:
    for clave in reg['medicion']:
        conteo[clave] = conteo.get(clave, 0) + 1

print('Presencia de cada campo dentro de medicion:')
for clave, n in sorted(conteo.items()):
    print(f'  {clave:20s} {n:5d} de {len(brutos)} registros')

Presencia de cada campo dentro de medicion:
  humedad_pct           2920 de 2920 registros
  indice_uv              736 de 2920 registros
  precipitacion_mm      2920 de 2920 registros
  temp_c                 736 de 2920 registros
  temperatura_c         2184 de 2920 registros


In [ ]:
filas = []
for reg in brutos:
    m = reg['medicion']
    filas.append({
        'region': reg['estacion']['region'],
        'codigo_estacion': reg['estacion']['codigo'],
        'fecha': reg['fecha'],
        'humedad_pct': m.get('humedad_pct'),
        'precipitacion_mm': m.get('precipitacion_mm'),
        # Resolución de la deriva: se acepta cualquiera de los dos nombres
        'temperatura_c': m.get('temperatura_c', m.get('temp_c')),
        'indice_uv': m.get('indice_uv'),  # None antes de la deriva
        '_esquema': 'v2' if 'temp_c' in m else 'v1',
    })

# infer_schema_length=None obliga a Polars a revisar TODAS las filas antes
# de decidir los tipos. Sin esto, indice_uv (nulo en las primeras 2184
# filas) provoca un error de inferencia de tipo.
clima = (pl.DataFrame(filas, infer_schema_length=None)
          .with_columns(pl.col('fecha').str.to_date()))

print(clima.group_by('_esquema')
      .agg(pl.len().alias('registros'))
      .sort('_esquema'))
clima.head(3)

shape: (2, 2)
┌──────────┬───────────┐
│ _esquema ┆ registros │
│ ---      ┆ ---       │
│ str      ┆ u32       │
╞══════════╪═══════════╡
│ v1       ┆ 2184      │
│ v2       ┆ 736       │
└──────────┴───────────┘


region,codigo_estacion,fecha,humedad_pct,precipitacion_mm,temperatura_c,indice_uv,_esquema
str,str,date,i64,f64,f64,i64,str
"""Lima""","""EST-01""",2025-09-01,59,0.0,15.0,null,"""v1"""
"""Lima""","""EST-01""",2025-09-02,63,4.7,16.5,null,"""v1"""
"""Lima""","""EST-01""",2025-09-03,66,1.1,16.9,null,"""v1"""


In [ ]:
aterrizar_bronce(clima, 'clima_diario', DATOS / 'clima_diario.jsonl')

  bronce <- clima_diario: 2,920 filas en lakehouse/bronce/clima_diario/fecha_carga=2026-09-02/clima_diario.parquet


PosixPath('lakehouse/bronce/clima_diario/fecha_carga=2026-09-02/clima_diario.parquet')

### PREGUNTA

El DataFrame `clima` debe tener 2 920 filas, la columna `temperatura_c` no debe tener nulos, y `indice_uv` debe tener exactamente 2 184 nulos.

Si usted fuera el consumidor de esta fuente, ¿qué cláusula habría incluido en el contrato de datos para que este cambio no lo hubiera tomado por sorpresa?

_Respuesta: Habría incluido una cláusula de versionado y notificación de esquema en el contrato de datos, que exigiera al proveedor: declarar explícitamente todos los nombres de campo válidos para cada variable (aceptando alias conocidos, como temperatura_c/temp_c, en lugar de que aparezca uno nuevo sin aviso

## Paso 5 — Consumir una API pública con reintentos

**Qué se busca:** Consumir el servicio de series estadísticas del BCRP aplicando tiempo de espera explícito, reintentos con retroceso exponencial y degradación controlada hacia un respaldo local.

In [ ]:
def obtener_json(url, intentos=3, espera_base=2.0, timeout=20):
    """Cliente HTTP con reintentos y retroceso exponencial."""
    ultimo_error = None
    for n in range(1, intentos + 1):
        try:
            t0 = time.time()
            resp = requests.get(url, timeout=timeout)
            resp.raise_for_status()
            meta = {
                'url': resp.url,
                'estado_http': resp.status_code,
                'segundos': round(time.time() - t0, 2),
                'intento': n,
            }
            return resp.json(), meta
        except requests.RequestException as e:
            ultimo_error = e
            print(f'  intento {n} fallido: {e.__class__.__name__}')
            if n < intentos:
                espera = espera_base ** n
                print(f'  reintentando en {espera:.1f} s...')
                time.sleep(espera)
    raise RuntimeError(f'Sin respuesta de {url}') from ultimo_error

In [ ]:
SERIE = 'PN01207PD'  # tipo de cambio bancario promedio, venta, diario
desde, hasta = date(2025, 9, 1), date(2026, 8, 31)
url = (f'https://estadisticas.bcrp.gob.pe/estadisticas/series/api/'
       f'{SERIE}/json/{desde:%Y-%m-%d}/{hasta:%Y-%m-%d}/ing')

try:
    payload, meta = obtener_json(url)
    tipo_cambio = pl.DataFrame([
        {'periodo': p['name'], 'valor': p['values'][0]}
        for p in payload['periods']
    ])
    print('API en vivo OK:', tipo_cambio.height, 'períodos')
    print('Metadatos:', json.dumps(meta, indent=2, ensure_ascii=False))
    origen_tc = 'API BCRP en vivo'
except Exception as e:
    print(f'API no disponible ({type(e).__name__}). Se usa el respaldo local.')
    tipo_cambio = pl.read_csv(DATOS / 'bcrp_tipo_cambio.csv',
                                try_parse_dates=True)
    origen_tc = 'respaldo local'

print(f'\nOrigen efectivo: {origen_tc}')
tipo_cambio.head(5)

  intento 1 fallido: JSONDecodeError
  reintentando en 2.0 s...
  intento 2 fallido: JSONDecodeError
  reintentando en 4.0 s...
  intento 3 fallido: JSONDecodeError
API no disponible (RuntimeError). Se usa el respaldo local.

Origen efectivo: respaldo local


fecha,tipo_cambio_venta
date,f64
2025-09-01,3.7175
2025-09-02,3.7241
2025-09-03,3.7281
2025-09-04,3.7257
2025-09-05,3.7261


In [ ]:
aterrizar_bronce(tipo_cambio, 'bcrp_tipo_cambio',
                   DATOS / 'bcrp_tipo_cambio.csv')

  bronce <- bcrp_tipo_cambio: 261 filas en lakehouse/bronce/bcrp_tipo_cambio/fecha_carga=2026-09-02/bcrp_tipo_cambio.parquet


PosixPath('lakehouse/bronce/bcrp_tipo_cambio/fecha_carga=2026-09-02/bcrp_tipo_cambio.parquet')

###PREGUNTA

 En el cuaderno queda registrado si obtuvo el dato de la API en vivo o del respaldo local. Ambos resultados son válidos.

Explique por qué la espera entre reintentos crece de forma exponencial en lugar de ser constante, y qué problema causaría un reintento inmediato en bucle contra un servicio saturado.

_Respuesta: La espera crece de forma exponencial porque, si un servicio está fallando o saturado, un tiempo de espera constante y corto seguiría golpeándolo casi a la misma frecuencia, ; en cambio, aumentar la espera en cada intento da al servidor cada vez más tiempo para liberar recursos

## Paso 6 — Declarar el contrato de datos y validarlo


In [ ]:
# Se lee desde BRONCE, no desde el CSV. A partir de aquí, el archivo
# original ya no vuelve a tocarse.
ventas_b = pl.read_parquet(BRONCE / 'ventas_pos' / '**' / '*.parquet')
print(f'Leídas {ventas_b.height:,} filas desde la zona bronce')

CONTRATO_VENTAS = {
    'id_venta': pl.Int64,
    'fecha': pl.Date,
    'id_local': pl.Int64,
    'id_producto': pl.Int64,
    'cantidad': pl.Int64,
    'importe': pl.Float64,
    'canal': pl.String,
    'medio_pago': pl.String,
}

Leídas 200,600 filas desde la zona bronce


In [ ]:
def validar_contrato(df, contrato, nombre):
    """Verifica presencia de columnas y compatibilidad de tipos."""
    faltantes = [c for c in contrato if c not in df.columns]
    nuevas = [c for c in df.columns
              if c not in contrato and not c.startswith('_')]
    tipos = [f'{c}: esperado {t}, recibido {df.schema[c]}'
             for c, t in contrato.items()
             if c in df.columns and df.schema[c] != t]

    if nuevas:
        print(f'  AVISO — deriva en {nombre}: columnas nuevas {nuevas}')
    if faltantes or tipos:
        raise ValueError(f'Contrato incumplido en {nombre}. '
                          f'Faltantes: {faltantes}. Tipos: {tipos}')
    print(f'  Contrato OK en {nombre}: '
          f'{df.height:,} filas x {df.width} columnas')
    return True

validar_contrato(ventas_b, CONTRATO_VENTAS, 'ventas_pos')

  Contrato OK en ventas_pos: 200,600 filas x 11 columnas


True

In [ ]:
REGLAS = {
    'importe_nulo': pl.col('importe').is_null(),
    'fecha_futura': pl.col('fecha') > pl.lit(date(2026, 8, 31)),
    'cantidad_cero': pl.col('cantidad') == 0,
}

diagnostico = ventas_b.select([
    expr.sum().alias(nombre) for nombre, expr in REGLAS.items()
])
print('Registros que violan cada regla:')
print(diagnostico)

duplicados = ventas_b.select(pl.exclude('^_.*$')).is_duplicated().sum()
print(f'\nFilas involucradas en duplicados exactos: {duplicados:,}')

Registros que violan cada regla:
shape: (1, 3)
┌──────────────┬──────────────┬───────────────┐
│ importe_nulo ┆ fecha_futura ┆ cantidad_cero │
│ ---          ┆ ---          ┆ ---           │
│ u32          ┆ u32          ┆ u32           │
╞══════════════╪══════════════╪═══════════════╡
│ 4012         ┆ 180          ┆ 0             │
└──────────────┴──────────────┴───────────────┘

Filas involucradas en duplicados exactos: 1,200


In [ ]:
sospechoso = (REGLAS['importe_nulo']
              | REGLAS['fecha_futura']
              | REGLAS['cantidad_cero'])

cuarentena = ventas_b.filter(sospechoso)
aceptadas = ventas_b.filter(~sospechoso)

cuarentena.write_parquet(CUARENTENA / 'ventas_pos_rechazadas.parquet')

tasa = 100 * cuarentena.height / ventas_b.height
print(f'Cuarentena: {cuarentena.height:,} filas ({tasa:.2f} %)')
print(f'Aceptadas : {aceptadas.height:,} filas')

Cuarentena: 4,189 filas (2.09 %)
Aceptadas : 196,411 filas


###PREGUNTA
196 413 aceptadas. Observe que 4 010 + 180 no suma 4 187: la diferencia se debe a los duplicados, que multiplican algunas filas defectuosas.

Explique por qué las devoluciones (cantidad negativa) NO fueron enviadas a cuarentena pese a ser valores atípicos.

_Respuesta: Las devoluciones no fueron enviadas a cuarentena porque,  tienen cantidad negativa y son valores atípicos frente al resto de las transacciones, no violan ninguna de las tres reglas de calidad declaradas

## Paso 7 — Normalizar y construir la zona plata


In [ ]:
plata = (aceptadas
    # 1) Deduplicar por las columnas de NEGOCIO, no por las de procedencia
    .unique(subset=['id_venta', 'fecha', 'id_local', 'id_producto',
                     'cantidad', 'importe'], keep='first')
    # 2) Normalizar texto y derivar columnas de análisis
    .with_columns([
        pl.col('canal').str.strip_chars().str.to_titlecase().alias('canal'),
        pl.col('medio_pago').str.strip_chars()
            .str.to_titlecase().alias('medio_pago'),
        (pl.col('cantidad') < 0).alias('es_devolucion'),
        pl.col('fecha').dt.year().alias('anio'),
        pl.col('fecha').dt.month().alias('mes'),
    ])
    # 3) Orden explícito de columnas
    .select(['id_venta', 'fecha', 'anio', 'mes', 'id_local', 'id_producto',
              'cantidad', 'importe', 'canal', 'medio_pago', 'es_devolucion'])
)

print('Canales tras normalizar:', sorted(plata['canal'].unique().to_list()))
print('Devoluciones marcadas :', plata['es_devolucion'].sum())
print('Filas en zona plata   :', f'{plata.height:,}')

Canales tras normalizar: ['Delivery', 'Presencial', 'Web']
Devoluciones marcadas : 2937
Filas en zona plata   : 195,823


In [ ]:
destino_plata = PLATA / 'ventas'
plata.write_parquet(destino_plata,
                      partition_by=['anio', 'mes'],
                      compression='zstd')

# Inspeccionar la estructura de directorios generada
carpetas = sorted(p for p in destino_plata.rglob('anio=*/mes=*') if p.is_dir())
print(f'Particiones creadas: {len(carpetas)}')
for c in carpetas[:5]:
    print('  ', c.relative_to(destino_plata))
print('  ...')

Particiones creadas: 12
   anio=2025/mes=10
   anio=2025/mes=11
   anio=2025/mes=12
   anio=2025/mes=9
   anio=2026/mes=1
  ...


## Paso 8 — Medir el efecto del formato columnar


In [ ]:
def tam_mb(ruta):
    """Tamaño en MB de un archivo o del árbol completo de un directorio."""
    ruta = Path(ruta)
    if ruta.is_file():
        return ruta.stat().st_size / 1e6
    return sum(f.stat().st_size
               for f in ruta.rglob('*') if f.is_file()) / 1e6

mb_csv = tam_mb(DATOS / 'ventas_pos.csv')
mb_pq = tam_mb(destino_plata)

print(f'CSV original       : {mb_csv:7.2f} MB')
print(f'Parquet particion.  : {mb_pq:7.2f} MB')
print(f'Reducción           : {mb_csv / mb_pq:7.1f}x')

CSV original       :   10.54 MB
Parquet particion.  :    1.99 MB
Reducción           :     5.3x


In [ ]:
con = duckdb.connect()

ruta_csv = str(DATOS / 'ventas_pos.csv')

# --- Consulta A: sobre Parquet particionado ---
t0 = time.time()
resultado_pq = con.execute(f"""
    SELECT id_local,
           ROUND(SUM(importe), 2) AS venta_total,
           COUNT(*) AS n_tickets
    FROM read_parquet('{destino_plata}/**/*.parquet', hive_partitioning=true)
    WHERE anio = 2026 AND mes = 3
    GROUP BY id_local
    ORDER BY venta_total DESC
    LIMIT 5
""").df()
t_pq = time.time() - t0

# --- Consulta B: sobre el CSV original ---
t0 = time.time()
resultado_csv = con.execute(f"""
    SELECT id_local,
           ROUND(SUM(importe), 2) AS venta_total,
           COUNT(*) AS n_tickets
    FROM read_csv_auto('{ruta_csv}')
    WHERE year(CAST(fecha AS DATE)) = 2026
      AND month(CAST(fecha AS DATE)) = 3
    GROUP BY id_local
    ORDER BY venta_total DESC
    LIMIT 5
""").df()
t_csv = time.time() - t0

print(f'Parquet particionado: {t_pq:.3f} s')
print(f'CSV completo        : {t_csv:.3f} s')
print(f'Aceleración         : {t_csv / t_pq:.1f}x')
resultado_pq

Parquet particionado: 0.012 s
CSV completo        : 0.261 s
Aceleración         : 21.3x


,id_local,venta_total,n_tickets
0,42,20005.54,171
1,83,18000.65,153
2,81,17818.89,152
3,120,17416.45,151
4,86,17369.00,147


### Registro obligatorio de resultados

Complete la tabla siguiente con los valores obtenidos en su equipo. La columna de referencia corresponde a una ejecución del docente y sirve solo como orden de magnitud esperado.

| Métrica | Referencia del docente | Su valor |
|---|---|---|
| Tamaño del CSV original | 10,54 MB |10,54 MB |
| Tamaño del Parquet particionado | 1,98 MB | 1,99 MB|
| Factor de reducción de tamaño | 5,3x | 5,3x|
| Tiempo de consulta sobre Parquet | 0,014 s |0,009 s |
| Tiempo de consulta sobre CSV | 0,167 s | 0,225 s |
| Factor de aceleración | 11,6x |26,0x |

**Punto de control 8.** Explique por qué la aceleración es mayor que la reducción de tamaño. Pista: la consulta filtra por un solo mes y usa solo tres de las once columnas disponibles.

_Respuesta: La aceleración es mayor que la reducción de tamaño porque el formato Parquet no solo pesa menos en disco, sino que además permite  optimizaciones que el CSV no tiene. Poda de particiones: al filtrar WHERE anio = 2026 AND mes = 3, DuckDB puede descartar directamente los directorios de las demás combinaciones año/mes sin siquiera abrirlos, mientras que en el CSV debe leer el archivo completo (los 12 meses) y recién después filtrar fila por fila.


## Paso 9 — Consulta analítica de verificación



In [ ]:
resumen_canal = con.execute(f"""
    SELECT canal,
           COUNT(*) AS tickets,
           ROUND(SUM(importe), 2) AS importe_total,
           ROUND(AVG(importe), 2) AS ticket_promedio
    FROM read_parquet('{destino_plata}/**/*.parquet', hive_partitioning=true)
    WHERE NOT es_devolucion
    GROUP BY canal
    ORDER BY importe_total DESC
""").df()
resumen_canal

,canal,tickets,importe_total,ticket_promedio
0,Presencial,138670,14748446.99,106.36
1,Delivery,34726,3688078.43,106.21
2,Web,19490,2067639.39,106.09


Se cumple
1. El manifiesto tiene al menos cinco líneas(15 actualmente)
2. Existe el archivo de cuarentena
3. La carpeta de plata tiene 12 particiones
4. La consulta de verificación devuelve exactamente tres canales (Presencial, delivery, web)


## 5. Actividad propuesta


### Tarea 1 — Claves huérfanas de producto

In [ ]:

maestro_productos_b = pl.read_parquet(BRONCE / 'maestro_productos' / '**' / '*.parquet')
locales_b = pl.read_parquet(BRONCE / 'locales' / '**' / '*.parquet')

# how='anti' devuelve las filas de la izquierda (plata) que NO tienen
# correspondencia en la derecha (maestro de productos)
huerfanos = plata.join(maestro_productos_b, on='id_producto', how='anti')

n_ventas_huerfanas = huerfanos.height
n_codigos_huerfanos = huerfanos['id_producto'].n_unique()

print(f'Ventas con id_producto huérfano            : {n_ventas_huerfanas:,}')
print(f'Códigos de producto distintos sin maestro  : {n_codigos_huerfanos}')

Ventas con id_producto huérfano            : 341
Códigos de producto distintos sin maestro  : 98


### Tarea 2 — Tabla enriquecida (unión con maestros)

In [ ]:
# Tarea 2: Tabla enriquecida (unión con maestros)
# Se usa LEFT JOIN, conservan ventas huerfanas y quedan con 'categoria' nula,
# visible y auditable.
enriquecida = (
    plata
    .join(maestro_productos_b.select(['id_producto', 'categoria']),
          on='id_producto', how='left')
    .join(locales_b.select(['id_local', 'region']),
          on='id_local', how='left')
)

print(f'Filas en tabla enriquecida : {enriquecida.height:,}')
print(f'Filas con categoría nula   : {enriquecida["categoria"].null_count():,}')
enriquecida.head(5)


Filas en tabla enriquecida : 195,823
Filas con categoría nula   : 341


id_venta,fecha,anio,mes,id_local,id_producto,cantidad,importe,canal,medio_pago,es_devolucion,categoria,region
i64,date,i32,i8,i64,i64,i64,f64,str,str,bool,str,str
146388,2025-09-13,2025,9,29,1087,6,56.51,"""Web""","""Tarjeta""",false,"""Analgésicos""","""Arequipa"""
45419,2026-08-05,2026,8,14,1046,6,37.11,"""Presencial""","""Efectivo""",false,"""Genéricos""","""Lima"""
133766,2025-10-30,2025,10,80,1294,5,175.94,"""Presencial""","""Tarjeta""",false,"""Antigripales""","""Arequipa"""
198480,2025-10-25,2025,10,118,1696,3,39.1,"""Delivery""","""Tarjeta""",false,"""Higiene personal""","""Lima"""
5754,2026-05-05,2026,5,30,1379,6,141.12,"""Presencial""","""Tarjeta""",false,"""Cuidado infantil""","""Cusco"""


### Tarea 3 — Cruce con clima por región y fecha

In [ ]:

# La clave de unión es compuesta: region proviene del maestro de locales,

enriquecida_clima = enriquecida.join(
    clima.select(['region', 'fecha', 'temperatura_c']),
    on=['region', 'fecha'],
    how='left'
)

sin_temperatura = enriquecida_clima['temperatura_c'].null_count()
print(f'Filas sin temperatura tras cruzar con clima: {sin_temperatura:,}')

Filas sin temperatura tras cruzar con clima: 0


### Tarea 4 — Correlación temperatura–cantidad (Antigripales)

In [ ]:

# excluyendo devoluciones, agrupada por región
correlacion_antigripales = (
    enriquecida_clima
    .filter((pl.col('categoria') == 'Antigripales') & (~pl.col('es_devolucion')))
    .group_by('region')
    .agg(pl.corr('temperatura_c', 'cantidad').alias('correlacion'))
    .sort('correlacion')
)

correlacion_antigripales

region,correlacion
str,f64
"""Áncash""",-0.370439
"""Arequipa""",-0.355266
"""Lambayeque""",-0.349782
"""Lima""",-0.344865
"""Piura""",-0.339196
"""Junín""",-0.337174
"""La Libertad""",-0.330735
"""Cusco""",-0.330669


### Tarea 5 — Ficha de fuente y conclusión para la gerencia

| Indicador | Valor obtenido |  | Verificación |
|---|---|---|---|
| Ventas con id_producto huérfano | 341 |  | ✓ |
| Códigos de producto distintos sin maestro | 98 |  | ✓ |
| Filas con categoría nula tras la unión | 341 |  | ✓ |
| Filas sin temperatura tras cruzar con clima | 0 |  | ✓ |
| Correlación temperatura–cantidad en Antigripales | Entre −0,33 y −0,38 en las 8 regiones |  | ✓ |

### Conclusión para gerencia

La demanda de antigripales muestra una correlación negativa y consistente con la temperatura en las ocho regiones analizadas (entre −0,33 y −0,38): a menor temperatura, mayor volumen de ventas. Se recomienda anticipar el refuerzo de inventario de esta categoría en las regiones donde se pronostiquen caídas de temperatura antes del próximo invierno, priorizando aquellas con la correlación más fuerte (Áncash, Arequipa y Lambayeque). Cabe advertir que esta correlación no implica causalidad directa: la temperatura es probablemente un indicador indirecto de la estacionalidad respiratoria, más que su causa.

## Anexo B. Ficha de fuente de datos — clima_diario.jsonl

| Campo | Contenido |
|---|---|
| Nombre de la fuente | clima_diario.jsonl |
| Entidad responsable | Servicio meteorológico (fuente externa) |
| Tipo (origen / estructura) | Externa · semiestructurada (JSON Lines, con anidamiento en `estacion` y `medicion`) |
| Modo de acceso y patrón de ingesta | Archivo entregado por lote (batch), leído línea por línea con `json.loads()` y aterrizado en la zona bronce mediante `aterrizar_bronce` |
| Período cubierto y granularidad | Mediciones diarias por región, del 01/09/2025 al 31/08/2026 |
| Frecuencia de actualización | Diaria (una medición por región y día); en este laboratorio se recibe como snapshot estático |
| Fecha de obtención | Fecha de ejecución del Paso 4 (ver partición `fecha_carga=` generada en `lakehouse/bronce/clima_diario/`) |
| Volumen (filas y tamaño) | 2 920 registros, 0,46 MB |
| Licencia o base de licitud | Datos simulados con fines pedagógicos para este laboratorio |
| ¿Contiene datos personales? ¿Cuáles? | No. Contiene mediciones agregadas por región y fecha (humedad, precipitación, temperatura, índice UV); no incluye identificadores de personas |
| Tratamiento de anonimización aplicado | No aplica, al no contener datos personales |
| Problemas de calidad detectados | Deriva de esquema: a partir del 01/06/2026 el campo `temperatura_c` fue renombrado a `temp_c` y se agregó el nuevo campo `indice_uv`, sin aviso previo del proveedor |
| Decisión: se incorpora o se descarta, y por qué | Se incorpora. La deriva se resolvió unificando ambos nombres en una sola columna (`temperatura_c`) y documentando la versión de esquema de cada registro (`_esquema` = v1/v2), conservando la trazabilidad sin pérdida de información |